# Optimization Lab: Unconstrained + Lagrange Multipliers (Lecture 5)

Notebook นี้ให้คุณจำลองและเปรียบเทียบเนื้อหาจากเลคเชอร์ 5 ได้เอง:

1. **Single-variable optimization** — หา critical point ด้วย f'(x)=0 แล้วเช็ค f''(x)
2. **Multi-variable + Hessian test** — ฟังก์ชันทั่วไป: หา ∇f=0 แล้วจัด max/min/saddle อัตโนมัติ
3. **ตรวจคำตอบด้วย scipy.optimize** (numerical) เทียบกับ sympy (symbolic)
4. **Lagrange Multipliers** — ฟังก์ชันทั่วไปสำหรับ constrained optimization พร้อมตัวอย่างจากสไลด์ทั้ง 4 ข้อ
5. ช่องท้ายไฟล์ให้ลองใส่ฟังก์ชันของคุณเอง


In [ ]:
import sympy as sp
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

sp.init_printing()


## 1. Single-variable optimization

ตัวอย่างจากสไลด์: l(x) = x⁴ + 7x³ + 5x² − 17x + 3


### 📝 วิธีทำด้วยมือ (ละเอียด) — $l(x)=x^4+7x^3+5x^2-17x+3$

**หลักการ (First & Second Derivative Test)**

> จุดสุดขีดภายใน (interior extremum) ต้องเป็น **critical point** คือจุดที่ $l'(x)=0$ (หรือ $l'$ ไม่มีค่า)
> แล้วใช้ $l''$ ตัดสินว่าเป็นอะไร: $l''>0$ → โค้งหงาย → **min**, $l''<0$ → โค้งคว่ำ → **max**, $l''=0$ → **สรุปไม่ได้**

---

#### ขั้นที่ 1 — หา $l'(x)$

ใช้ power rule กับทุกพจน์: $\frac{d}{dx}x^n = nx^{n-1}$

$$l'(x) = 4x^3 + 21x^2 + 10x - 17$$

#### ขั้นที่ 2 — แก้ $l'(x)=0$

เป็นสมการดีกรี 3 ลองหารากตรรกยะก่อน (ตัวประกอบของ 17 หารด้วยตัวประกอบของ 4):

- $l'(1) = 4+21+10-17 = 18 \neq 0$
- $l'(-1) = -4+21-10-17 = -10 \neq 0$

→ **ไม่มีรากสวย** ต้องหาเชิงตัวเลข วิธีสอบมือคือ **ตารางเปลี่ยนเครื่องหมาย** (Intermediate Value Theorem):

| $x$ | $-5$ | $-4$ | $-2$ | $-1$ | $0$ | $1$ |
|---|---|---|---|---|---|---|
| $l'(x)$ | $-42$ | $+23$ | $+15$ | $-10$ | $-17$ | $+18$ |

เครื่องหมายพลิก 3 ครั้ง → มีราก 3 ตัว (ครบตามดีกรี ⇒ **รากจริงทั้งหมด**):

$$x_1 \in (-5,-4),\qquad x_2 \in (-2,-1),\qquad x_3 \in (0,1)$$

ซอยต่อด้วย bisection จะได้

$$\boxed{x_1 \approx -4.480,\quad x_2 \approx -1.432,\quad x_3 \approx 0.662}$$

**✅ เช็คคำตอบด้วย Vieta:** ผลบวกรากของ $4x^3+21x^2+10x-17$ ต้องเท่ากับ $-\tfrac{21}{4} = -5.25$
$$-4.480 + (-1.432) + 0.662 = -5.250 \;\checkmark$$

#### ขั้นที่ 3 — หา $l''(x)$ แล้วแทนค่า

$$l''(x) = 12x^2 + 42x + 10$$

| $x$ | $12x^2$ | $42x$ | $+10$ | $l''(x)$ | สรุป |
|---|---|---|---|---|---|
| $-4.480$ | $240.84$ | $-188.16$ | $10$ | $+62.68$ | $>0$ → **local min** |
| $-1.432$ | $24.61$ | $-60.14$ | $10$ | $-25.54$ | $<0$ → **local max** |
| $0.662$ | $5.26$ | $27.80$ | $10$ | $+43.06$ | $>0$ → **local min** |

#### ขั้นที่ 4 — หาค่า $l(x)$ ที่แต่ละจุด แล้วสรุป global

$$l(-4.480) \approx -47.08,\qquad l(-1.432) \approx +21.25,\qquad l(0.662) \approx -3.84$$

เนื่องจากดีกรีเป็น **เลขคู่** และสัมประสิทธิ์นำ $=+1>0$ ⇒ $l(x)\to+\infty$ ทั้งสองปลาย
⇒ **ไม่มี global maximum** และ global minimum คือจุดที่ต่ำสุดในบรรดา local min:

$$\textbf{Global min ที่ } x\approx-4.480,\ l\approx-47.08$$

> ⚠️ นี่คือเหตุผลที่เซลล์ถัดไปต้องรัน `scipy.minimize` จาก **หลาย initial point** — ฟังก์ชันนี้ไม่ convex, ถ้าเริ่มใกล้ $x=0.662$ มันจะติดอยู่ที่ local min ตัวนั้นและไม่เจอคำตอบจริง


In [ ]:
x = sp.symbols('x')
l = x**4 + 7*x**3 + 5*x**2 - 17*x + 3

dl = sp.diff(l, x)
d2l = sp.diff(l, x, 2)
print("l'(x)  =", dl)
print("l''(x) =", d2l)

critical_points_raw = sp.solve(sp.Eq(dl, 0), x)
# แปลงเป็นตัวเลขและกรองเฉพาะรากจริง (สมการดีกรี 3 ให้ค่าในรูป symbolic ที่อ่านยาก จึงประเมินเป็นตัวเลข)
critical_points = []
for cp in critical_points_raw:
    val = complex(cp.evalf())
    if abs(val.imag) < 1e-9:
        critical_points.append(sp.Float(val.real))
print("Critical points (real, ตัวเลข):", [round(float(c), 4) for c in critical_points])

for cp in critical_points:
    second = d2l.subs(x, cp)
    kind = "local minimum" if second > 0 else ("local maximum" if second < 0 else "inconclusive (ต้องเช็คเพิ่ม)")
    print(f"x = {float(cp):.4f} -> l''(x) = {float(second):.4f} -> {kind}")


In [ ]:
# วาดกราฟพร้อม critical points
f_num = sp.lambdify(x, l, 'numpy')
xs = np.linspace(-6, 2, 400)
ys = f_num(xs)

plt.figure(figsize=(6, 4))
plt.plot(xs, ys)
for cp in critical_points:
    if cp.is_real:
        cpf = float(cp)
        plt.plot(cpf, f_num(cpf), 'ro')
        plt.annotate(f"({cpf:.2f}, {f_num(cpf):.1f})", (cpf, f_num(cpf)), textcoords="offset points", xytext=(6, 6))
plt.xlabel("x")
plt.ylabel("l(x)")
plt.title("l(x) = x^4 + 7x^3 + 5x^2 - 17x + 3")
plt.grid(alpha=0.3)
plt.show()

# เทียบกับ scipy (numerical, ต้องลองหลาย initial point เพราะ non-convex)
for x0 in [-6, -2, 0, 2]:
    res = minimize(lambda v: f_num(v[0]), x0=[x0])
    print(f"start x0={x0:>3} -> scipy พบ x* = {res.x[0]:.4f}, f(x*) = {res.fun:.4f}")


## 2. Multi-variable: Hessian test (generic function)

ฟังก์ชันนี้ใช้ได้กับ f(x,y) ใดๆ: หา ∇f=0 ด้วย sympy, คำนวณ Hessian, แล้วจัดประเภทแต่ละ critical point อัตโนมัติ


### 📝 ทฤษฎีที่โค้ดข้างล่างทำ — Second Partial Derivative Test

**สูตรที่ต้องจำ (4 ขั้น)**

1. หา **gradient** แล้วตั้งให้เป็นศูนย์ทั้งสองตัว
$$\nabla f = \begin{pmatrix} f_x \\ f_y \end{pmatrix} = \begin{pmatrix} 0 \\ 0 \end{pmatrix}
\quad\Longrightarrow\quad \text{ระบบสมการ 2 ตัวแปร → critical points}$$

2. หาอนุพันธ์อันดับสองทั้งสามตัว: $f_{xx},\ f_{yy},\ f_{xy}$ (จำไว้ว่า $f_{xy}=f_{yx}$ ถ้าฟังก์ชันเรียบ — **Clairaut's theorem**)

3. สร้าง **Hessian** และหา **discriminant** $D$
$$H = \begin{pmatrix} f_{xx} & f_{xy} \\ f_{xy} & f_{yy}\end{pmatrix},
\qquad D \;=\; \det H \;=\; f_{xx}f_{yy} - (f_{xy})^2$$

4. แทนค่า critical point ลงใน $D$ แล้วอ่านตาราง:

| เงื่อนไข | ผลสรุป |
|---|---|
| $D > 0$ และ $f_{xx} > 0$ | **Local minimum** (ชามหงาย) |
| $D > 0$ และ $f_{xx} < 0$ | **Local maximum** (ชามคว่ำ) |
| $D < 0$ | **Saddle point** (อานม้า) |
| $D = 0$ | **สรุปไม่ได้** — test ใช้ไม่ได้ ต้องดูวิธีอื่น |

---

#### 💡 ทำไมสูตรนี้ใช้ได้ (เข้าใจแล้วไม่ต้องจำ)

$H$ เป็นเมทริกซ์สมมาตร มีค่าเจาะจง (eigenvalues) $\lambda_1,\lambda_2$ เป็นจำนวนจริง และ

$$\lambda_1\lambda_2 = \det H = D, \qquad \lambda_1+\lambda_2 = \operatorname{tr} H = f_{xx}+f_{yy}$$

$\lambda$ คือ "ความโค้ง" ของผิวในแต่ละแกนหลัก ดังนั้น

- $D<0$ ⇒ $\lambda_1,\lambda_2$ **คนละเครื่องหมาย** ⇒ แกนหนึ่งโค้งขึ้น อีกแกนโค้งลง ⇒ **saddle**
- $D>0$ ⇒ $\lambda$ **เครื่องหมายเดียวกัน** ⇒ โค้งไปทางเดียวกันทุกทิศ ⇒ เป็น max หรือ min
  แล้วดูว่าทางไหน: ถ้า $\lambda$ ทั้งคู่บวก ผลรวมต้องบวก และเพราะ $f_{xx}f_{yy}=D+f_{xy}^2>0$ จึงเช็คแค่ $f_{xx}$ ตัวเดียวก็พอ
- $D=0$ ⇒ มี $\lambda=0$ ⇒ มีทิศที่ "แบน" ⇒ อนุพันธ์อันดับสองบอกอะไรไม่ได้ ต้องดูอันดับสูงขึ้น

> 🔎 แนวเดียวกันนี้ขยายไป $n$ ตัวแปรได้: min ⇔ $H$ **positive definite**, max ⇔ **negative definite**, saddle ⇔ **indefinite**


In [ ]:
xs_, ys_ = sp.symbols('x y')

def analyze_critical_points(f_expr, variables=(xs_, ys_), verbose=True):
    """
    f_expr : sympy expression ของ f(x, y)
    คืนค่า list ของ dict {point, fxx, fyy, fxy, D, classification}
    """
    grad = [sp.diff(f_expr, v) for v in variables]
    solutions = sp.solve(grad, variables, dict=True)

    fxx = sp.diff(f_expr, variables[0], 2)
    fyy = sp.diff(f_expr, variables[1], 2)
    fxy = sp.diff(f_expr, variables[0], variables[1])

    results = []
    for sol in solutions:
        pt = tuple(sol[v] for v in variables)
        if not all(v.is_real for v in pt):
            continue
        Hxx = fxx.subs(sol)
        Hyy = fyy.subs(sol)
        Hxy = fxy.subs(sol)
        D = Hxx * Hyy - Hxy**2

        if D < 0:
            kind = "Saddle point"
        elif D > 0 and Hxx > 0:
            kind = "Local minimum"
        elif D > 0 and Hxx < 0:
            kind = "Local maximum"
        else:
            kind = "Inconclusive (D=0)"

        results.append({"point": pt, "fxx": Hxx, "fyy": Hyy, "fxy": Hxy, "D": D, "kind": kind})
        if verbose:
            print(f"Critical point {pt}: f_xx={Hxx}, f_yy={Hyy}, f_xy={Hxy}, D={D}  ->  {kind}")

    return results


In [ ]:
# ตัวอย่างสไลด์: f(x,y) = x^3 - y^3 + 9xy
f1 = xs_**3 - ys_**3 + 9*xs_*ys_
print("f(x,y) =", f1, "\n")
results1 = analyze_critical_points(f1)


### 📝 วิธีทำด้วยมือ — $f(x,y)=x^3-y^3+9xy$

**ขั้นที่ 1 — หา gradient แล้วตั้ง $=0$**

$$f_x = 3x^2 + 9y = 0 \quad\cdots(1) \qquad\qquad f_y = -3y^2 + 9x = 0 \quad\cdots(2)$$

**ขั้นที่ 2 — แก้ระบบด้วยการแทนค่า (substitution)**

จาก (1): $\;9y = -3x^2 \;\Rightarrow\; y = -\dfrac{x^2}{3}$

แทนใน (2): $\;-3\left(-\dfrac{x^2}{3}\right)^2 + 9x = 0 \;\Rightarrow\; -3\cdot\dfrac{x^4}{9} + 9x = 0 \;\Rightarrow\; -\dfrac{x^4}{3} + 9x = 0$

คูณ $-3$ ทั้งสองข้าง: $\;x^4 - 27x = 0 \;\Rightarrow\; x(x^3 - 27) = 0$

$$\Rightarrow\; x = 0 \quad\text{หรือ}\quad x = 3$$

ย้อนหา $y$ จาก $y=-x^2/3$:

- $x=0 \Rightarrow y = 0$ → จุด $(0,0)$
- $x=3 \Rightarrow y = -\dfrac{9}{3} = -3$ → จุด $(3,-3)$

> ⚠️ **กับดักที่เจอบ่อย:** $x^3=27$ ในจำนวนจริงให้ $x=3$ ตัวเดียว (อีกสองรากเป็นจำนวนเชิงซ้อน จึงทิ้ง) — นี่คือเหตุผลที่โค้ดมีบรรทัด `if not all(v.is_real ...)`

**ขั้นที่ 3 — อนุพันธ์อันดับสองและ $D$**

$$f_{xx} = 6x, \qquad f_{yy} = -6y, \qquad f_{xy} = 9$$

$$D = f_{xx}f_{yy} - f_{xy}^2 = (6x)(-6y) - 81 = \boxed{-36xy - 81}$$

**ขั้นที่ 4 — แทนค่าแต่ละจุด**

| จุด | $f_{xx}$ | $f_{yy}$ | $f_{xy}$ | $D=-36xy-81$ | สรุป |
|---|---|---|---|---|---|
| $(0,0)$ | $0$ | $0$ | $9$ | $-36(0)-81 = -81 < 0$ | **Saddle point** |
| $(3,-3)$ | $18$ | $18$ | $9$ | $-36(3)(-3)-81 = 324-81 = 243 > 0$ | $D>0,\ f_{xx}=18>0$ → **Local minimum** |

**ค่าฟังก์ชันที่จุด min:**
$$f(3,-3) = 27 - (-27) + 9(3)(-3) = 27 + 27 - 81 = \boxed{-27}$$

> 📌 สังเกต: local min นี้ **ไม่ใช่** global min เพราะพจน์ $x^3$ ทำให้ $f\to-\infty$ เมื่อ $x\to-\infty$ — ฟังก์ชันนี้ไม่มี global extremum


In [ ]:
# ตัวอย่างสไลด์: f(x,y) = 3y^2 - 2y^3 - 3x^2 + 6xy
f2 = 3*ys_**2 - 2*ys_**3 - 3*xs_**2 + 6*xs_*ys_
print("f(x,y) =", f2, "\n")
results2 = analyze_critical_points(f2)


### 📝 วิธีทำด้วยมือ — $f(x,y)=3y^2-2y^3-3x^2+6xy$

**ขั้นที่ 1 — gradient $=0$**

$$f_x = -6x + 6y = 0 \quad\cdots(1) \qquad\qquad f_y = 6y - 6y^2 + 6x = 0 \quad\cdots(2)$$

**ขั้นที่ 2 — แก้ระบบ**

จาก (1) หารด้วย 6: $\;-x + y = 0 \;\Rightarrow\; \boxed{y = x}$  ← สมการนี้ง่ายกว่า เริ่มจากตัวนี้เสมอ

แทนใน (2) แล้วหารด้วย 6: $\;y - y^2 + x = 0 \;\xrightarrow{x=y}\; y - y^2 + y = 0 \;\Rightarrow\; 2y - y^2 = 0$

$$y(2-y) = 0 \;\Rightarrow\; y = 0 \ \text{ หรือ } \ y = 2$$

ได้ critical points: $\;(0,0)\;$ และ $\;(2,2)$

**ขั้นที่ 3 — อนุพันธ์อันดับสองและ $D$**

$$f_{xx} = -6, \qquad f_{yy} = 6 - 12y, \qquad f_{xy} = 6$$

$$D = (-6)(6-12y) - 6^2 = -36 + 72y - 36 = \boxed{72y - 72 = 72(y-1)}$$

> 💡 น่าสังเกต: $D$ ขึ้นกับ $y$ เท่านั้น และพลิกเครื่องหมายที่ $y=1$ พอดี

**ขั้นที่ 4 — แทนค่า**

| จุด | $f_{xx}$ | $f_{yy}=6-12y$ | $f_{xy}$ | $D=72(y-1)$ | สรุป |
|---|---|---|---|---|---|
| $(0,0)$ | $-6$ | $6$ | $6$ | $72(-1) = -72 < 0$ | **Saddle point** |
| $(2,2)$ | $-6$ | $-18$ | $6$ | $72(1) = 72 > 0$ | $D>0,\ f_{xx}=-6<0$ → **Local maximum** |

**ค่าฟังก์ชันที่จุด max:**
$$f(2,2) = 3(4) - 2(8) - 3(4) + 6(2)(2) = 12 - 16 - 12 + 24 = \boxed{8}$$

> 📌 $f_{xx}=-6$ เป็นลบ **ทุกจุด** — แนวแกน $x$ โค้งคว่ำเสมอ สิ่งที่ตัดสินว่าเป็น saddle หรือ max คือความโค้งในแนว $y$ ($f_{yy}$) ที่พลิกเครื่องหมายเมื่อ $y$ ผ่าน $\tfrac12$


In [ ]:
# ตัวอย่างสไลด์: f(x,y) = x^2 + y^2 - 4x - 6y + 20  (ไม่มี saddle เพราะเป็น convex bowl)
f3 = xs_**2 + ys_**2 - 4*xs_ - 6*ys_ + 20
print("f(x,y) =", f3, "\n")
results3 = analyze_critical_points(f3)

# เทียบกับ scipy
f3_num = sp.lambdify((xs_, ys_), f3, 'numpy')
res = minimize(lambda v: f3_num(v[0], v[1]), x0=[0, 0])
print("\nscipy พบจุดต่ำสุดที่:", res.x, "ค่า f =", res.fun)


### 📝 วิธีทำด้วยมือ — $f(x,y)=x^2+y^2-4x-6y+20$

**ขั้นที่ 1–2 — gradient $=0$ (คราวนี้แยกกันสนิท ไม่ต้องแทนค่า)**

$$f_x = 2x - 4 = 0 \;\Rightarrow\; x = 2 \qquad\qquad f_y = 2y - 6 = 0 \;\Rightarrow\; y = 3$$

critical point เดียว: $(2,3)$

**ขั้นที่ 3–4 — Hessian**

$$f_{xx} = 2, \qquad f_{yy} = 2, \qquad f_{xy} = 0 \qquad\Rightarrow\qquad D = (2)(2) - 0^2 = 4 > 0$$

$D>0$ และ $f_{xx}=2>0$ → **Local minimum**

$$f(2,3) = 4 + 9 - 8 - 18 + 20 = \boxed{7}$$

---

#### ✅ วิธีเช็คแบบไม่ใช้อนุพันธ์เลย — Completing the square

$$
\begin{aligned}
f &= (x^2 - 4x) + (y^2 - 6y) + 20 \\
  &= (x^2 - 4x + 4) - 4 + (y^2 - 6y + 9) - 9 + 20 \\
  &= (x-2)^2 + (y-3)^2 + 7
\end{aligned}
$$

เพราะกำลังสองทั้งสองก้อน $\ge 0$ เสมอ ⇒ $f \ge 7$ ทุก $(x,y)$ และเท่ากับ 7 **เฉพาะ** ตอน $x=2,y=3$

$$\Rightarrow (2,3) \text{ เป็น } \textbf{global minimum} \text{ ไม่ใช่แค่ local}$$

> 📌 นี่คือความหมายของคำว่า **convex**: Hessian $H=\begin{pmatrix}2&0\\0&2\end{pmatrix}$ เป็น positive definite **ทุกจุด** (ไม่ใช่แค่ที่ critical point) ⇒ ฟังก์ชันเป็นชามหงายทั้งผิว ⇒ มี critical point ได้แค่จุดเดียว, ไม่มี saddle, ไม่มี local max, และ local min = global min
>
> ⇒ `scipy.minimize` เริ่มจากจุดไหนก็เจอ $(2,3)$ เหมือนกันหมด ต่างจากตัวอย่างข้อ 1 ที่ต้องลองหลาย initial point


### Plot แบบหมุนได้ (interactive)

`matplotlib` แบบ static หมุนไม่ได้ในบางสภาพแวดล้อม (เช่น VS Code / Colab บาง cell) จึงเปลี่ยนมาใช้
`plotly` ซึ่งให้กราฟที่ **ลากเมาส์หมุนได้จริง** ในเอาต์พุตของ notebook โดยตรง (ไม่ต้องติดตั้งอะไรเพิ่มถ้ามี
`plotly` อยู่แล้ว — ถ้ายังไม่มีให้รัน `pip install plotly` ก่อน)


In [ ]:
import plotly.graph_objects as go

def plot_surface_3d(f_expr, results, xr=(-5, 5), yr=(-5, 5), title=""):
    f_num = sp.lambdify((xs_, ys_), f_expr, 'numpy')
    X, Y = np.meshgrid(np.linspace(*xr, 60), np.linspace(*yr, 60))
    Z = f_num(X, Y)

    fig = go.Figure()

    # พื้นผิวหลัก
    fig.add_trace(go.Surface(
        x=X, y=Y, z=Z,
        colorscale="Viridis",
        opacity=0.9,
        showscale=False,
        contours={"z": {"show": True, "usecolormap": True, "project": {"z": True}}},
    ))

    # จุด critical points แต่ละประเภท ใช้สีต่างกัน
    colors = {"Local minimum": "blue", "Local maximum": "red", "Saddle point": "black"}
    seen_kinds = set()
    for r in results:
        px, py = float(r["point"][0]), float(r["point"][1])
        pz = float(f_num(px, py))
        kind = r["kind"]
        fig.add_trace(go.Scatter3d(
            x=[px], y=[py], z=[pz],
            mode="markers",
            marker={"size": 6, "color": colors.get(kind, "gray")},
            name=kind,
            showlegend=(kind not in seen_kinds),
        ))
        seen_kinds.add(kind)

    fig.update_layout(
        title=title,
        scene={"xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "f(x,y)"},
        margin={"l": 0, "r": 0, "t": 40, "b": 0},
        legend={"x": 0.02, "y": 0.98},
        width=650, height=550,
    )
    fig.show()
    return fig

plot_surface_3d(f1, results1, xr=(-6, 6), yr=(-6, 6), title="f(x,y) = x^3 - y^3 + 9xy")
plot_surface_3d(f2, results2, xr=(-4, 5), yr=(-4, 5), title="f(x,y) = 3y^2 - 2y^3 - 3x^2 + 6xy")


## 3. Lagrange Multipliers (generic function)

แก้ปัญหา: optimize f(x,y) subject to g(x,y) = 0 โดยตั้งสมการ ∇f = λ∇g และ g = 0 แล้วให้ sympy แก้ระบบสมการ


### 📝 ทฤษฎี Lagrange Multipliers — ทำไมต้อง $\nabla f = \lambda \nabla g$

**ปัญหา:** หา max/min ของ $f(x,y)$ **โดยมีเงื่อนไข** $g(x,y)=0$ (จุดคำตอบต้องอยู่บนเส้น/เส้นโค้ง $g=0$)

#### 💡 สัญชาตญาณเชิงเรขาคณิต

วาด **level curves** ของ $f$ (เส้น $f=c$ หลาย ๆ ค่า) ทับกับเส้นเงื่อนไข $g=0$

- ถ้าเส้น $f=c$ **ตัด** เส้น $g=0$ แบบตัดผ่าน → เดินไปตามเส้น $g=0$ ต่ออีกหน่อยจะข้ามไป $f=c+\varepsilon$ ได้ ⇒ **ยังไม่สุด**
- จุดที่สุดจริง คือจุดที่เส้น $f=c$ **สัมผัส** (tangent) กับเส้น $g=0$ พอดี

จุดสัมผัส ⇒ เส้นสัมผัสของทั้งสองเส้นทับกัน ⇒ **เส้นตั้งฉาก** ของทั้งสองขนานกัน และ gradient คือเวกเตอร์ตั้งฉากของ level curve จึงได้

$$\boxed{\nabla f = \lambda \nabla g}$$

$\lambda$ (Lagrange multiplier) คือ "ตัวคูณปรับความยาว" เพราะสองเวกเตอร์ขนานกันแต่ยาวไม่เท่ากัน

#### 📐 สูตรใช้งาน — ระบบ 3 สมการ 3 ตัวไม่รู้ค่า

$$
\begin{cases}
f_x = \lambda\, g_x & \cdots(1)\\
f_y = \lambda\, g_y & \cdots(2)\\
g(x,y) = 0 & \cdots(3) \quad \leftarrow \textbf{ห้ามลืมสมการนี้!}
\end{cases}
$$

หรือเขียนรวบเป็นฟังก์ชันเดียว (**Lagrangian**) แล้วหา critical point ของมันแบบ unconstrained:

$$\mathcal{L}(x,y,\lambda) = f(x,y) - \lambda\, g(x,y), \qquad \nabla \mathcal{L} = 0$$

#### 🔧 เทคนิคแก้ระบบ (ลำดับที่ควรทำ)

1. **หาร (1)÷(2)** เพื่อกำจัด $\lambda$ ทันที → ได้ความสัมพันธ์ระหว่าง $x,y$
   ⚠️ ระวัง: ใช้ได้เมื่อ $g_y \ne 0$ — ต้องแยกเคส $g_y=0$ ตรวจด้วย
2. หรือ **ย้าย $\lambda$ ออกมา** จาก (1) และ (2) แล้วจับให้เท่ากัน
3. เอาผลจากข้อ 1 แทนใน **(3)** → เหลือสมการตัวแปรเดียว → แก้ได้
4. ย้อนกลับหาตัวแปรที่เหลือ **ทุกกรณี** (อย่าลืมรากลบ / $\pm$)

#### ⚠️ จุดที่นักเรียนพลาดบ่อยที่สุด

> **Lagrange บอกแค่ "จุดผู้สมัคร" (candidates) — ไม่บอกว่าอันไหน max อันไหน min!**

ต้องตัดสินเอง เลือกวิธีใดวิธีหนึ่ง:

| สถานการณ์ | วิธีตัดสิน |
|---|---|
| เงื่อนไขเป็นเส้นโค้ง **ปิดและมีขอบเขต** (วงกลม, วงรี) | เป็นเซต compact ⇒ มี max และ min แน่นอน ⇒ **แทนค่า $f$ ทุก candidate แล้วเทียบตัวเลข** |
| เงื่อนไขเป็น **เส้นตรง** (ไม่ compact) | **แทนค่าลดตัวแปร** ให้เหลือ 1 ตัวแปร แล้วใช้ $f''$ ตัดสิน (ดูตัวอย่าง 3, 4) |
| ทั่วไป | ใช้ **bordered Hessian** |

#### 💰 ความหมายของ $\lambda$ (shadow price)

$$\lambda = \frac{\partial f^*}{\partial b} \quad \text{เมื่อเงื่อนไขคือ } g(x,y) = b$$

แปลว่า **ถ้าคลายเงื่อนไขเพิ่ม 1 หน่วย ค่าที่ดีที่สุดจะดีขึ้นประมาณ $\lambda$ หน่วย** — ในโจทย์เศรษฐศาสตร์คือ "ราคาเงา" ของทรัพยากร (ดูตัวอย่าง 4)


In [ ]:
lam = sp.symbols('lambda')

def lagrange_solve(f_expr, g_expr, variables=(xs_, ys_), verbose=True):
    """
    f_expr : objective function f(x, y)
    g_expr : constraint ในรูป g(x, y) = 0
    คืนค่า list ของ dict {point, f_value}
    """
    grad_f = [sp.diff(f_expr, v) for v in variables]
    grad_g = [sp.diff(g_expr, v) for v in variables]

    equations = [sp.Eq(grad_f[i], lam * grad_g[i]) for i in range(len(variables))]
    equations.append(sp.Eq(g_expr, 0))

    solutions = sp.solve(equations, list(variables) + [lam], dict=True)

    results = []
    for sol in solutions:
        pt = tuple(sol[v] for v in variables)
        if not all(v.is_real for v in pt):
            continue
        f_val = f_expr.subs(sol)
        results.append({"point": pt, "f_value": f_val})
        if verbose:
            print(f"(x, y) = ({pt[0]}, {pt[1]})  ->  f = {f_val}   (lambda = {sol[lam]})")

    return results


In [ ]:
# Example 1: f(x,y) = xy, g: x^2/8 + y^2/2 - 1 = 0
f_e1 = xs_ * ys_
g_e1 = xs_**2 / 8 + ys_**2 / 2 - 1
print("=== Example 1: f=xy บนวงรี x^2/8 + y^2/2 = 1 ===")
res_e1 = lagrange_solve(f_e1, g_e1)


### 📝 วิธีทำด้วยมือ — Example 1: $f=xy$ บนวงรี $\dfrac{x^2}{8}+\dfrac{y^2}{2}=1$

**ขั้นที่ 1 — เขียน $g$ ให้เป็น $g=0$ แล้วหา gradient ทั้งสอง**

$$g(x,y) = \frac{x^2}{8} + \frac{y^2}{2} - 1$$

$$\nabla f = \begin{pmatrix} y \\ x \end{pmatrix}, \qquad
\nabla g = \begin{pmatrix} \frac{2x}{8} \\ \frac{2y}{2} \end{pmatrix} = \begin{pmatrix} \frac{x}{4} \\ y \end{pmatrix}$$

**ขั้นที่ 2 — ตั้งระบบสมการ**

$$
\begin{cases}
y = \lambda \cdot \dfrac{x}{4} & \cdots(1)\\[4pt]
x = \lambda y & \cdots(2)\\[4pt]
\dfrac{x^2}{8} + \dfrac{y^2}{2} = 1 & \cdots(3)
\end{cases}
$$

**ขั้นที่ 3 — กำจัด $\lambda$**

แทน (2) ลงใน (1): $\;y = \dfrac{\lambda(\lambda y)}{4} = \dfrac{\lambda^2 y}{4}$

ย้ายข้าง: $\;y\left(1 - \dfrac{\lambda^2}{4}\right) = 0 \;\Rightarrow\; y=0 \ \text{ หรือ } \ \lambda^2 = 4$

**🔍 ตรวจเคส $y=0$ ก่อน (ห้ามข้าม):** ถ้า $y=0$ แล้ว (2) ให้ $x=\lambda\cdot 0 = 0$ ⇒ จุด $(0,0)$
แต่แทนใน (3): $\;0+0 = 0 \ne 1$ ⇒ **ไม่อยู่บนวงรี ⇒ ตกไป** ✗

ดังนั้น $\;\lambda = \pm 2$

**ขั้นที่ 4 — แยกสองเคสของ $\lambda$ แล้วแทนใน (3)**

🔹 **เคส $\lambda = 2$:** จาก (2) $\;x = 2y$

แทนใน (3): $\;\dfrac{(2y)^2}{8} + \dfrac{y^2}{2} = \dfrac{4y^2}{8} + \dfrac{y^2}{2} = \dfrac{y^2}{2} + \dfrac{y^2}{2} = y^2 = 1 \;\Rightarrow\; y = \pm 1$

- $y=1 \Rightarrow x=2$ → $(2,\,1)$
- $y=-1 \Rightarrow x=-2$ → $(-2,\,-1)$

🔹 **เคส $\lambda = -2$:** จาก (2) $\;x = -2y$

แทนใน (3): $\;\dfrac{4y^2}{8} + \dfrac{y^2}{2} = y^2 = 1 \;\Rightarrow\; y = \pm 1$

- $y=1 \Rightarrow x=-2$ → $(-2,\,1)$
- $y=-1 \Rightarrow x=2$ → $(2,\,-1)$

**ขั้นที่ 5 — เทียบค่า $f=xy$ ของทั้ง 4 candidates**

| จุด | $\lambda$ | $f = xy$ | สรุป |
|---|---|---|---|
| $(2,\,1)$ | $2$ | $+2$ | **Maximum** |
| $(-2,\,-1)$ | $2$ | $+2$ | **Maximum** |
| $(-2,\,1)$ | $-2$ | $-2$ | **Minimum** |
| $(2,\,-1)$ | $-2$ | $-2$ | **Minimum** |

$$\boxed{f_{\max} = 2 \text{ ที่ } (\pm2,\pm1)\ \text{(เครื่องหมายเดียวกัน)}, \qquad f_{\min} = -2 \text{ ที่ } (\mp2,\pm1)}$$

> ✅ **มั่นใจได้ว่านี่คือ global** เพราะวงรีเป็นเซต **compact** (ปิด + มีขอบเขต) และ $f$ ต่อเนื่อง ⇒ Extreme Value Theorem รับประกันว่า max/min มีอยู่จริงและต้องอยู่ในบรรดา candidates เหล่านี้
>
> 📌 ความสมมาตร: $f=xy$ ไม่เปลี่ยนค่าเมื่อสลับ $(x,y)\to(-x,-y)$ และวงรีก็สมมาตรแบบเดียวกัน ⇒ คำตอบจึงมาเป็นคู่เสมอ


In [ ]:
# Example 2: f(x,y) = 3x + 4y, g: x^2 + y^2 - 1 = 0
f_e2 = 3*xs_ + 4*ys_
g_e2 = xs_**2 + ys_**2 - 1
print("=== Example 2: f=3x+4y บนวงกลม x^2+y^2=1 ===")
res_e2 = lagrange_solve(f_e2, g_e2)


### 📝 วิธีทำด้วยมือ — Example 2: $f=3x+4y$ บนวงกลม $x^2+y^2=1$

**ขั้นที่ 1 — gradients**

$$g = x^2+y^2-1, \qquad \nabla f = \begin{pmatrix}3\\4\end{pmatrix}, \qquad \nabla g = \begin{pmatrix}2x\\2y\end{pmatrix}$$

**ขั้นที่ 2 — ระบบสมการ**

$$
\begin{cases}
3 = 2\lambda x & \cdots(1)\\
4 = 2\lambda y & \cdots(2)\\
x^2+y^2 = 1 & \cdots(3)
\end{cases}
$$

**ขั้นที่ 3 — แก้ $x,y$ ในรูป $\lambda$**

สังเกตว่า $\lambda \ne 0$ (ถ้า $\lambda=0$ สมการ (1) จะกลายเป็น $3=0$ ซึ่งเป็นไปไม่ได้) ⇒ หารได้

$$x = \frac{3}{2\lambda}, \qquad y = \frac{4}{2\lambda} = \frac{2}{\lambda}$$

**ขั้นที่ 4 — แทนใน (3) หา $\lambda$**

$$\left(\frac{3}{2\lambda}\right)^2 + \left(\frac{2}{\lambda}\right)^2 = 1
\;\Rightarrow\; \frac{9}{4\lambda^2} + \frac{4}{\lambda^2} = 1
\;\Rightarrow\; \frac{9 + 16}{4\lambda^2} = 1$$

$$\Rightarrow\; 4\lambda^2 = 25 \;\Rightarrow\; \lambda = \pm\frac{5}{2}$$

**ขั้นที่ 5 — ย้อนหาจุดและเทียบค่า**

🔹 $\lambda = \frac{5}{2}$: $\;x = \dfrac{3}{2 \cdot 5/2} = \dfrac{3}{5}, \quad y = \dfrac{2}{5/2} = \dfrac{4}{5}$

$$f = 3\left(\tfrac35\right) + 4\left(\tfrac45\right) = \tfrac{9}{5} + \tfrac{16}{5} = \tfrac{25}{5} = 5$$

🔹 $\lambda = -\frac{5}{2}$: $\;x = -\dfrac{3}{5}, \quad y = -\dfrac{4}{5}$

$$f = -\tfrac95 - \tfrac{16}{5} = -5$$

$$\boxed{f_{\max} = 5 \text{ ที่ } \left(\tfrac35, \tfrac45\right), \qquad f_{\min} = -5 \text{ ที่ } \left(-\tfrac35, -\tfrac45\right)}$$

---

#### ✅ เช็คด้วย Cauchy–Schwarz (ไม่ต้องใช้แคลคูลัสเลย)

$f = 3x+4y$ คือ dot product $\mathbf{a}\cdot\mathbf{v}$ โดย $\mathbf{a}=(3,4)$, $\mathbf{v}=(x,y)$ ที่มี $\|\mathbf{v}\|=1$

$$|\mathbf{a}\cdot\mathbf{v}| \le \|\mathbf{a}\|\,\|\mathbf{v}\| = \sqrt{3^2+4^2}\cdot 1 = 5$$

เท่ากันเมื่อ $\mathbf{v}$ **ขนานกับ** $\mathbf{a}$ นั่นคือ $\mathbf{v} = \pm\dfrac{(3,4)}{5} = \pm\left(\tfrac35,\tfrac45\right)$ ✓ **ตรงกันเป๊ะ**

> 💡 นี่คือกรณีที่เห็นภาพ Lagrange ชัดที่สุด: level curve ของ $f=3x+4y$ คือ **เส้นตรงขนานกัน** ค่อย ๆ เลื่อนออกไป จุดสุดท้ายที่เส้นยังแตะวงกลมคือจุด **สัมผัส** และที่จุดสัมผัสนั้น รัศมี $(x,y)$ ตั้งฉากกับเส้น ⇒ ขนานกับ $\nabla f=(3,4)$ พอดี
>
> 💰 $\lambda = \tfrac52$ แปลว่า: ถ้าขยายรัศมีวงกลมจาก $x^2+y^2=1$ เป็น $=1.1$ ค่า max จะเพิ่มขึ้นประมาณ $\tfrac52 \times 0.1 = 0.25$ (ของจริง: $5\sqrt{1.1}-5 = 0.244$ ✓ ใกล้มาก)


In [ ]:
# Example 3: f(x,y) = x^2 + 4y^2 - 2x + 8y, g: x + 2y - 7 = 0
f_e3 = xs_**2 + 4*ys_**2 - 2*xs_ + 8*ys_
g_e3 = xs_ + 2*ys_ - 7
print("=== Example 3: f=x^2+4y^2-2x+8y, constraint x+2y=7 ===")
res_e3 = lagrange_solve(f_e3, g_e3)


### 📝 วิธีทำด้วยมือ — Example 3: $f=x^2+4y^2-2x+8y$ ภายใต้ $x+2y=7$

**ขั้นที่ 1 — gradients**

$$g = x + 2y - 7, \qquad
\nabla f = \begin{pmatrix} 2x-2 \\ 8y+8 \end{pmatrix}, \qquad
\nabla g = \begin{pmatrix} 1 \\ 2 \end{pmatrix}$$

**ขั้นที่ 2 — ระบบสมการ**

$$
\begin{cases}
2x - 2 = \lambda \cdot 1 & \cdots(1)\\
8y + 8 = \lambda \cdot 2 & \cdots(2)\\
x + 2y = 7 & \cdots(3)
\end{cases}
$$

**ขั้นที่ 3 — แก้ $x,y$ ในรูป $\lambda$** (เงื่อนไขเป็นเส้นตรง จึงตรงไปตรงมา)

จาก (1): $\;x = \dfrac{\lambda+2}{2}$

จาก (2): $\;8y = 2\lambda - 8 \;\Rightarrow\; y = \dfrac{2\lambda-8}{8} = \dfrac{\lambda-4}{4}$

**ขั้นที่ 4 — แทนใน (3)**

$$\frac{\lambda+2}{2} + 2\cdot\frac{\lambda-4}{4} = 7
\;\Rightarrow\; \frac{\lambda+2}{2} + \frac{\lambda-4}{2} = 7
\;\Rightarrow\; \frac{2\lambda - 2}{2} = 7$$

$$\Rightarrow\; \lambda - 1 = 7 \;\Rightarrow\; \boxed{\lambda = 8}$$

**ขั้นที่ 5 — ย้อนหา $x,y$**

$$x = \frac{8+2}{2} = 5, \qquad y = \frac{8-4}{4} = 1$$

**ตรวจเงื่อนไข:** $\;5 + 2(1) = 7$ ✓

$$f(5,1) = 25 + 4(1) - 10 + 8 = \boxed{27}$$

---

#### ⚠️ แต่นี่เป็น max หรือ min? — Lagrange ไม่ได้บอก!

เงื่อนไขคือ **เส้นตรง** (ไม่ compact) จึงเทียบค่าไม่ได้เพราะมี candidate แค่จุดเดียว → ใช้ **วิธีลดตัวแปร**

แก้ (3) ให้ $x = 7 - 2y$ แล้วแทนกลับใน $f$:

$$
\begin{aligned}
\tilde f(y) &= (7-2y)^2 + 4y^2 - 2(7-2y) + 8y \\
&= (49 - 28y + 4y^2) + 4y^2 - 14 + 4y + 8y \\
&= 8y^2 - 16y + 35
\end{aligned}
$$

$$\tilde f'(y) = 16y - 16 = 0 \;\Rightarrow\; y = 1 \;\checkmark \text{ (ตรงกับ Lagrange)}$$

$$\tilde f''(y) = 16 > 0 \;\Rightarrow\; \textbf{Minimum}$$

$$\tilde f(1) = 8 - 16 + 35 = 27 \;\checkmark$$

$$\Rightarrow \boxed{(5,1) \text{ เป็น global minimum, } f_{\min}=27}$$

ไม่มี maximum เพราะเดินไปตามเส้นตรงไกล ๆ แล้ว $\tilde f(y) = 8y^2-\ldots \to +\infty$

> 📌 **วิธีลดตัวแปร vs Lagrange:** เมื่อ constraint เป็นเชิงเส้น ลดตัวแปรมักเร็วกว่า และได้ผลตัดสิน max/min มาฟรี ๆ
> แต่ Lagrange ชนะเมื่อ constraint แก้หา $x$ ในรูป $y$ ไม่ได้ง่าย ๆ (เช่น $x^2/8+y^2/2=1$ ที่ต้องใส่ $\pm\sqrt{\ }$) และขยายไป $n$ ตัวแปร / หลาย constraint ได้ทันที

---

#### 🎁 โบนัส — ทดสอบความหมายของ $\lambda$ (shadow price)

$\lambda=8$ ทำนายว่า ถ้าคลายเงื่อนไขเป็น $x+2y=8$ (เพิ่ม 1 หน่วย) ค่า $f_{\min}$ จะเพิ่มขึ้น $\approx 8$

**ลองทำจริง:** $x = 8-2y \Rightarrow \tilde f(y) = 8y^2 - 20y + 48$, $\;\tilde f' = 16y-20 = 0 \Rightarrow y = 1.25$

$$f_{\min}^{\text{new}} = 8(1.5625) - 25 + 48 = 35.5$$

เพิ่มขึ้นจริง $35.5 - 27 = 8.5$ เทียบกับที่ทำนาย $8$ → **ใกล้แต่ไม่เป๊ะ** เพราะ $\lambda$ เป็นอัตราเปลี่ยนแปลง **ณ จุดนั้น** (อนุพันธ์อันดับหนึ่ง) แต่การเพิ่มทีเดียว 1 หน่วยมีผลอันดับสองเข้ามาด้วย

**ตรวจแบบทั่วไป** — ให้เงื่อนไขเป็น $x+2y=b$ แล้วลดตัวแปรจะได้

$$f^*(b) = b^2 - 2b - \frac{(b-3)^2}{2}
\qquad\Longrightarrow\qquad
\frac{df^*}{db} = 2b - 2 - (b-3) = b + 1$$

ที่ $b=7$: $\;\dfrac{df^*}{db} = 8 = \lambda$ ✓ **ตรงตามทฤษฎีเป๊ะ**


In [ ]:
# Example 4 (golf ball profit): f(x,y) = 48x + 96y - x^2 - 2xy - 9y^2, g: 20x + 4y - 216 = 0
f_e4 = 48*xs_ + 96*ys_ - xs_**2 - 2*xs_*ys_ - 9*ys_**2
g_e4 = 20*xs_ + 4*ys_ - 216
print("=== Example 4: Pro-T golf ball profit ===")
res_e4 = lagrange_solve(f_e4, g_e4)


### 📝 วิธีทำด้วยมือ — Example 4: Pro-T golf ball profit

**โจทย์:** กำไร $P(x,y) = 48x + 96y - x^2 - 2xy - 9y^2$ (พันบาท) ภายใต้งบประมาณ $20x + 4y = 216$

**ขั้นที่ 0 — ลดรูปเงื่อนไขก่อน (ช่วยให้เลขง่ายขึ้นเยอะ)**

$$20x + 4y = 216 \;\xrightarrow{\ \div 4\ }\; 5x + y = 54$$

> 💡 ใช้ $g = 20x+4y-216$ หรือ $g=5x+y-54$ ก็ได้คำตอบ $(x,y)$ เดียวกัน — ต่างกันแค่ค่า $\lambda$ (ต่างกัน 4 เท่า) เพราะ $\lambda$ ขึ้นกับสเกลของ $g$

**ขั้นที่ 1 — gradients** (ใช้ $g = 20x+4y-216$ ตามโจทย์เดิม)

$$\nabla P = \begin{pmatrix} 48 - 2x - 2y \\ 96 - 2x - 18y \end{pmatrix}, \qquad
\nabla g = \begin{pmatrix} 20 \\ 4 \end{pmatrix}$$

> ⚠️ **จุดพลาดบ่อย:** พจน์ $-2xy$ ให้ $\partial/\partial x = -2y$ และ $\partial/\partial y = -2x$ — ต้องมีอยู่ในทั้งสองสมการ อย่าลืม

**ขั้นที่ 2 — ระบบสมการ**

$$
\begin{cases}
48 - 2x - 2y = 20\lambda & \cdots(1)\\
96 - 2x - 18y = 4\lambda & \cdots(2)\\
20x + 4y = 216 & \cdots(3)
\end{cases}
$$

**ขั้นที่ 3 — กำจัด $\lambda$**

จาก (1): $\;\lambda = \dfrac{48-2x-2y}{20} = \dfrac{24-x-y}{10}$

แทนใน (2):
$$96 - 2x - 18y = 4 \cdot \frac{24-x-y}{10} = \frac{2(24-x-y)}{5}$$

คูณ 5 ทั้งสองข้าง:
$$480 - 10x - 90y = 48 - 2x - 2y$$

ย้ายข้าง:
$$480 - 48 = 10x - 2x + 90y - 2y \;\Rightarrow\; 432 = 8x + 88y$$

หาร 8:
$$\boxed{x + 11y = 54} \quad \cdots(4)$$

**ขั้นที่ 4 — แก้ (3) กับ (4) พร้อมกัน**

จาก (3) รูปลดแล้ว: $\;y = 54 - 5x$

แทนใน (4):
$$x + 11(54 - 5x) = 54 \;\Rightarrow\; x + 594 - 55x = 54 \;\Rightarrow\; -54x = -540$$

$$\boxed{x = 10} \qquad\Rightarrow\qquad y = 54 - 5(10) = \boxed{4}$$

**ตรวจเงื่อนไข:** $\;20(10) + 4(4) = 200 + 16 = 216$ ✓

**ขั้นที่ 5 — หา $\lambda$ และกำไร**

$$\lambda = \frac{24 - 10 - 4}{10} = \frac{10}{10} = 1$$

$$
\begin{aligned}
P(10,4) &= 48(10) + 96(4) - (10)^2 - 2(10)(4) - 9(4)^2 \\
&= 480 + 384 - 100 - 80 - 144 \\
&= \boxed{540}
\end{aligned}
$$

---

#### ✅ ยืนยันว่าเป็น maximum (ไม่ใช่ min) ด้วยวิธีลดตัวแปร

แทน $y = 54 - 5x$ ลงใน $P$ โดยตรง:

$$
\begin{aligned}
\tilde P(x) &= 48x + 96(54-5x) - x^2 - 2x(54-5x) - 9(54-5x)^2 \\
&= 48x + 5184 - 480x - x^2 - 108x + 10x^2 - 9(2916 - 540x + 25x^2) \\
&= 48x + 5184 - 480x - x^2 - 108x + 10x^2 - 26244 + 4860x - 225x^2
\end{aligned}
$$

รวมพจน์:

| พจน์ | รวม |
|---|---|
| $x^2$ | $-1 + 10 - 225 = -216$ |
| $x$ | $48 - 480 - 108 + 4860 = 4320$ |
| ค่าคงที่ | $5184 - 26244 = -21060$ |

$$\tilde P(x) = -216x^2 + 4320x - 21060$$

$$\tilde P'(x) = -432x + 4320 = 0 \;\Rightarrow\; x = 10 \;\checkmark$$

$$\tilde P''(x) = -432 < 0 \;\Rightarrow\; \textbf{Maximum แน่นอน}$$

$$\tilde P(10) = -21600 + 43200 - 21060 = 540 \;\checkmark$$

---

#### 💰 ตีความเชิงธุรกิจ

| สิ่งที่ได้ | ค่า | ความหมาย |
|---|---|---|
| $x^*$ | $10$ | ผลิตสินค้า A 10 หน่วย |
| $y^*$ | $4$ | ผลิตสินค้า B 4 หน่วย |
| $P^*$ | $540$ | กำไรสูงสุด 540 (พันบาท) |
| $\lambda$ | $1$ | **ถ้าเพิ่มงบจาก 216 → 217 กำไรจะเพิ่มขึ้น $\approx 1$** |

$\lambda = 1 > 0$ ⇒ คุ้มที่จะขอเพิ่มงบ (ถ้าต้นทุนของงบเพิ่ม 1 หน่วยน้อยกว่า 1) — นี่คือเหตุผลที่นักเศรษฐศาสตร์เรียก $\lambda$ ว่า **marginal value / shadow price** ของทรัพยากร

> 🔎 **ลองเทียบดู:** ถ้าไม่มีข้อจำกัดงบเลย (unconstrained) จุด max อยู่ที่ไหน?
> $P_x = 48-2x-2y=0$ และ $P_y = 96-2x-18y=0$ → ลบกัน: $48 - 16y = 0 \Rightarrow y=3$, แล้ว $x = 24-3 = 21$
> ได้ $(21,3)$ กำไร $P = 48(21)+96(3)-441-126-81 = 1008+288-648 = 648$
> แต่จุดนี้ใช้งบ $20(21)+4(3) = 432 > 216$ ⇒ **เกินงบ** เงื่อนไขจึงมีผลจริง (active) และดึงกำไรลงจาก 648 → 540


In [ ]:
# เทียบกับ scipy.optimize.minimize (SLSQP รองรับ equality constraint) สำหรับ Example 4
f_e4_num = sp.lambdify((xs_, ys_), -f_e4, 'numpy')  # ใส่ลบเพราะ scipy minimize
constraint = {'type': 'eq', 'fun': lambda v: 20*v[0] + 4*v[1] - 216}

res = minimize(lambda v: f_e4_num(v[0], v[1]), x0=[5, 5], constraints=[constraint])
print("scipy (SLSQP):", res.x, "-> profit =", -res.fun)
print("sympy (Lagrange):", res_e4)


## 4. ลองฟังก์ชันของคุณเอง

แก้ `f_custom` และ `g_custom` ด้านล่างแล้วรันใหม่ — ใช้ได้ทั้ง unconstrained (Hessian test) และ constrained (Lagrange)


In [ ]:
# ==== Unconstrained: แก้ f_custom แล้วรัน ====
f_custom = xs_**2 * ys_ - 3*xs_*ys_**2 + ys_**3

print("Unconstrained analysis:")
results_custom = analyze_critical_points(f_custom)


### 📝 วิเคราะห์ตัวอย่าง `f_custom` ข้างบน — กรณีที่ Hessian test **ใช้ไม่ได้**

$$f(x,y) = x^2y - 3xy^2 + y^3$$

**ขั้นที่ 1–2 — หา critical points**

$$f_x = 2xy - 3y^2 = y(2x - 3y) = 0 \quad\cdots(1)$$
$$f_y = x^2 - 6xy + 3y^2 = 0 \quad\cdots(2)$$

สมการ (1) แยกตัวประกอบได้ ⇒ แตกเป็น 2 เคส:

🔹 **เคส A: $y = 0$**
แทนใน (2): $\;x^2 - 0 + 0 = 0 \Rightarrow x = 0$ → จุด $(0,0)$

🔹 **เคส B: $2x = 3y$** นั่นคือ $x = \tfrac{3y}{2}$
แทนใน (2):
$$\left(\tfrac{3y}{2}\right)^2 - 6\left(\tfrac{3y}{2}\right)y + 3y^2
= \tfrac{9y^2}{4} - 9y^2 + 3y^2
= \left(\tfrac94 - 6\right)y^2 = -\tfrac{15}{4}y^2 = 0$$
$$\Rightarrow y = 0 \Rightarrow x = 0 \quad\text{→ จุด } (0,0) \text{ เดิม}$$

$$\Rightarrow \textbf{มี critical point เดียวคือ } (0,0)$$

**ขั้นที่ 3–4 — Hessian**

$$f_{xx} = 2y, \qquad f_{yy} = -6x + 6y, \qquad f_{xy} = 2x - 6y$$

ที่ $(0,0)$ ทุกตัวเป็น **ศูนย์**:

$$f_{xx}=0,\quad f_{yy}=0,\quad f_{xy}=0 \qquad\Rightarrow\qquad D = 0\cdot 0 - 0^2 = 0$$

> ⛔ **$D=0$ ⇒ Second Derivative Test สรุปอะไรไม่ได้เลย** — โค้ดจึงพิมพ์ `Inconclusive (D=0)`
> เหตุผล: ทุกพจน์ของ $f$ มีดีกรี 3 ⇒ อนุพันธ์อันดับสองทุกตัวมีดีกรี 1 ⇒ เป็นศูนย์ที่ origin หมด ⇒ **ผิวแบนราบถึงอันดับสอง** ต้องดูอันดับสาม

---

#### 🔧 วิธีสรุปเมื่อ $D=0$ — ตรวจ $f$ ตามเส้นตรงผ่านจุดนั้น

เทคนิค: เดินเข้าหา $(0,0)$ ตามทิศต่าง ๆ แล้วดูเครื่องหมายของ $f$

🔹 **ตามเส้น $y = x$:**
$$f(x,x) = x^3 - 3x^3 + x^3 = -x^3$$

- $x > 0 \Rightarrow f = -x^3 < 0$ (ต่ำกว่า $f(0,0)=0$)
- $x < 0 \Rightarrow f = -x^3 > 0$ (สูงกว่า $f(0,0)=0$)

**ในทิศทางเดียวกัน ค่า $f$ ทั้งสูงกว่าและต่ำกว่า 0 ได้** (แค่คนละข้างของ origin)

$$\Rightarrow \boxed{(0,0) \textbf{ ไม่ใช่ทั้ง local max และ local min — เป็น degenerate saddle}}$$

🔹 **เช็คซ้ำด้วยการแยกตัวประกอบ** — ฟังก์ชันนี้แยกได้สวย:
$$f(x,y) = y(x^2 - 3xy + y^2)$$
ใกล้ origin พจน์ $y$ พลิกเครื่องหมายเมื่อข้ามแกน $x$ ⇒ $f$ พลิกเครื่องหมายด้วย ⇒ ยืนยันว่าไม่ใช่ extremum ✓

---

#### 📋 สรุปเช็กลิสต์เมื่อเจอ $D=0$

1. ลองแทน $f$ ตามเส้นตรงหลาย ๆ ทิศ ($y=0$, $x=0$, $y=x$, $y=-x$, $y=kx$) — ถ้าเจอ **สองทิศที่เครื่องหมายต่างกัน** ⇒ ไม่ใช่ extremum จบเลย
2. ถ้าทุกทิศเครื่องหมายเหมือนกัน **ยังสรุปไม่ได้** (อาจมีทิศโค้งที่ยังไม่ลอง — เช่น $y=x^2$) ต้องดูต่อ
3. ลอง **แยกตัวประกอบ / completing the square** เพื่อดูเครื่องหมายของ $f$ โดยตรง
4. หรือใช้ **Taylor expansion อันดับสูงขึ้น** ที่จุดนั้น


In [ ]:
# ==== Constrained (Lagrange): แก้ f_custom2 / g_custom แล้วรัน ====
f_custom2 = xs_**2 + ys_**2
g_custom = xs_ + ys_ - 4

print("Constrained (Lagrange) analysis:")
res_custom = lagrange_solve(f_custom2, g_custom)


### 📝 วิธีทำด้วยมือ — Lagrange (custom): $f=x^2+y^2$ ภายใต้ $x+y=4$

**ความหมายเชิงเรขาคณิต:** $f=x^2+y^2$ คือ **กำลังสองของระยะทาง** จากจุด $(x,y)$ ไปยังจุดกำเนิด $(0,0)$
ดังนั้นโจทย์นี้แท้จริงคือ "หาจุดบนเส้นตรง $x+y=4$ ที่อยู่ **ใกล้จุดกำเนิดที่สุด**"

**ขั้นที่ 1 — gradients**

$$g = x+y-4, \qquad \nabla f = \begin{pmatrix}2x\\2y\end{pmatrix}, \qquad \nabla g = \begin{pmatrix}1\\1\end{pmatrix}$$

**ขั้นที่ 2 — ระบบสมการ**

$$
\begin{cases}
2x = \lambda \cdot 1 & \cdots(1)\\
2y = \lambda \cdot 1 & \cdots(2)\\
x+y = 4 & \cdots(3)
\end{cases}
$$

**ขั้นที่ 3 — กำจัด $\lambda$**

จาก (1) และ (2): ทั้งคู่เท่ากับ $\lambda$ ⇒ $\;2x = 2y \;\Rightarrow\; \boxed{x=y}$

**ขั้นที่ 4 — แทนใน (3)**

$$x + x = 4 \;\Rightarrow\; 2x = 4 \;\Rightarrow\; \boxed{x = 2} \qquad\Rightarrow\qquad y = 2$$

ได้ candidate เดียว: $(2,2)$ พร้อม $\lambda$ จาก (1): $\;\lambda = 2(2) = 4$

$$f(2,2) = 4+4 = \boxed{8}$$

---

#### ⚠️ เป็น max หรือ min? — เงื่อนไขเป็นเส้นตรง (ไม่ compact) ต้องใช้วิธีลดตัวแปร

แก้ (3) ให้ $y = 4-x$ แล้วแทนกลับใน $f$:

$$
\begin{aligned}
\tilde f(x) &= x^2 + (4-x)^2 \\
&= x^2 + 16 - 8x + x^2 \\
&= 2x^2 - 8x + 16
\end{aligned}
$$

$$\tilde f'(x) = 4x - 8 = 0 \;\Rightarrow\; x = 2 \;\checkmark \text{ (ตรงกับ Lagrange)}$$

$$\tilde f''(x) = 4 > 0 \;\Rightarrow\; \textbf{Minimum แน่นอน}$$

$$\tilde f(2) = 8 - 16 + 16 = 8 \;\checkmark$$

เมื่อ $x\to\pm\infty$ ค่า $\tilde f(x) = 2x^2-\ldots \to +\infty$ ⇒ **ไม่มี maximum** บนเส้นตรงนี้ (เดินไปไกล ๆ ค่ายิ่งมากขึ้นเรื่อย ๆ)

$$\Rightarrow \boxed{(2,2) \text{ เป็น global minimum บนเส้น } x+y=4,\ f_{\min}=8}$$

---

#### ✅ ตรวจด้วยสูตรระยะทางจากจุดถึงเส้นตรง (ไม่ต้องใช้ Lagrange เลย)

ระยะทางจากจุดกำเนิด $(0,0)$ ถึงเส้น $x+y-4=0$ คือ

$$d = \frac{|1(0)+1(0)-4|}{\sqrt{1^2+1^2}} = \frac{4}{\sqrt2} = 2\sqrt2$$

และ $f_{\min}$ ควรเท่ากับ $d^2$ พอดี (เพราะ $f=x^2+y^2$ คือระยะทางกำลังสอง):

$$d^2 = (2\sqrt2)^2 = 8 \;\checkmark \text{ ตรงกับ } f_{\min}=8$$

> 📌 **ทำไม $x=y$ เสมอสำหรับปัญหานี้:** เพราะ $\nabla f=(2x,2y)$ ต้องขนานกับ $\nabla g=(1,1)$ ซึ่งมีสองพิกัดเท่ากัน ⇒ จุดที่ใกล้เส้นตรงที่สุดจากจุดกำเนิดต้องอยู่บนเส้นตั้งฉากที่ลากผ่านจุดกำเนิด — เป็นภาพเรขาคณิตเดียวกับสูตรระยะทางจุดถึงเส้นที่เรียนมาตั้งแต่ ม.ปลาย

#### 💰 ความหมายของ $\lambda=4$

ถ้าคลายเงื่อนไขจาก $x+y=4$ เป็น $x+y=4+\varepsilon$ ค่า $f_{\min}$ จะเพิ่มขึ้นประมาณ $4\varepsilon$

**ตรวจทั่วไป:** ให้ $x+y=b$ แล้วลดตัวแปรแบบเดียวกัน จะได้ $f^*(b) = \dfrac{b^2}{2}$ (พิสูจน์ได้จาก $\tilde f(x)=2x^2-2bx+b^2$ ที่จุดต่ำสุด $x=b/2$)

$$\frac{df^*}{db} = b \qquad \text{ที่ } b=4:\ \frac{df^*}{db}=4=\lambda \;\checkmark \text{ ตรงตามทฤษฎีเป๊ะ}$$


---

## 📊 Visualization: Unconstrained vs Constrained Optimum

### ตัวอย่าง 1: $f=x^2+4y^2-2x+8y$ ก่อนและหลัง Lagrange

**วัตถุประสงค์:** เห็นภาพความแตกต่างระหว่างจุด optimum เมื่อ
1. **ไม่มีเงื่อนไข (unconstrained)** — ศูนย์สูตร = ทุกทิศ
2. **มีเงื่อนไข $x+2y=7$ (constrained)** — ต้องอยู่บนเส้นตรง

**กรณีศึกษาเหมาะสมแบบนี้เพราะ:**
- Unconstrained: $(2,3)$, $f=7$ (global min, สวยสมบูรณ์)
- Constrained: $(5,1)$, $f=27$ (บนเส้น $x+2y=7$ เท่านั้น, ห่างจากจริง)
- แลขอบเขตของตัวแปรจึงเห็นว่า constraint ดึงคำตอบไปไหน

**2D Contour Plot:** แสดงเส้น level curve ของ $f$ + เส้น constraint


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Example 3: Unconstrained vs Constrained
# f(x,y) = x^2 + 4y^2 - 2x + 8y  with  constraint: x + 2y = 7

f3 = xs_**2 + 4*ys_**2 - 2*xs_ + 8*ys_
f3_num = sp.lambdify((xs_, ys_), f3, 'numpy')

xr, yr = np.linspace(-2, 10, 200), np.linspace(-2, 8, 200)
X, Y = np.meshgrid(xr, yr)
Z = f3_num(X, Y)

unconstrained_pt = (2, 3)
constrained_pt = (5, 1)
f_unconstrained = 7
f_constrained = 27

fig = make_subplots(rows=1, cols=2,
                     subplot_titles=("ไม่มีเงื่อนไข (Unconstrained)",
                                      "มีเงื่อนไข x+2y=7 (Constrained)"))

for col in [1, 2]:
    fig.add_trace(go.Contour(
        x=xr, y=yr, z=Z,
        colorscale="RdYlGn_r", opacity=0.75, showscale=False,
        contours={"coloring": "lines", "showlabels": True},
        line={"width": 1.5},
    ), row=1, col=col)

fig.add_trace(go.Scatter(
    x=[unconstrained_pt[0]], y=[unconstrained_pt[1]], mode="markers",
    marker={"symbol": "star", "size": 16, "color": "blue"},
    name=f"Unconstrained {unconstrained_pt}, f={f_unconstrained}",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=[unconstrained_pt[0]], y=[unconstrained_pt[1]], mode="markers",
    marker={"symbol": "star", "size": 16, "color": "blue"},
    name=f"Unconstrained {unconstrained_pt}, f={f_unconstrained}",
    showlegend=False,
), row=1, col=2)

y_constraint = np.linspace(-1, 6, 100)
x_constraint = 7 - 2 * y_constraint
fig.add_trace(go.Scatter(
    x=x_constraint, y=y_constraint, mode="lines",
    line={"color": "red", "width": 3},
    name="Constraint: x+2y=7",
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=[constrained_pt[0]], y=[constrained_pt[1]], mode="markers",
    marker={"symbol": "circle", "size": 12, "color": "green"},
    name=f"Constrained {constrained_pt}, f={f_constrained}",
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=[unconstrained_pt[0], constrained_pt[0]],
    y=[unconstrained_pt[1], constrained_pt[1]],
    mode="lines", line={"color": "purple", "width": 2, "dash": "dash"},
    name="Unconstrained -> Constrained", showlegend=False,
), row=1, col=2)

fig.update_xaxes(title_text="x", range=[-2, 10])
fig.update_yaxes(title_text="y", range=[-2, 8])
fig.update_layout(
    title="Level Curves: f(x,y) = x^2 + 4y^2 - 2x + 8y",
    width=950, height=460,
    legend={"orientation": "h", "y": -0.15},
)
fig.show()

print("สังเกต:")
print(f"  - Unconstrained min: (2, 3), f = 7")
print(f"  - Constrained min: (5, 1), f = 27")
print(f"  - ค่า f เพิ่มขึ้นจาก 7 -> 27 เพราะ constraint บังคับให้เคลื่อนออกจากจุด global min")
print(f"  - ที่ constrained optimum: gradient f ขนานกับ gradient g (tangent condition)")


---

### ตัวอย่าง 2: $P=48x+96y-x^2-2xy-9y^2$ (Pro-T Golf) ก่อนและหลัง Lagrange

**กรณีศึกษานี้ดูเสมือนสถานการณ์จริง:**
- Unconstrained: $(21,3)$, $P=648$ (กำไรสูงสุดถ้าไม่จำกัดงบ)
- Constrained: $(10,4)$, $P=540$ (เมื่องบจำกัดที่ 216)
- **ขาดกำไรไป 108** (= 648-540) เพราะต้องผลิตน้อยกว่า

**ภาพประกอบ:** 3D surface plot + constraint plane + ทั้งสองจุด


In [ ]:
from plotly.subplots import make_subplots

# Example 4: Pro-T Golf Ball -- Unconstrained vs Constrained
# P(x,y) = 48x + 96y - x^2 - 2xy - 9y^2  with  constraint: 20x + 4y = 216

P = 48*xs_ + 96*ys_ - xs_**2 - 2*xs_*ys_ - 9*ys_**2
P_num = sp.lambdify((xs_, ys_), P, 'numpy')

unconstrained_pt = (21, 3)
constrained_pt = (10, 4)
P_unconstrained = 648
P_constrained = 540
budget_check_unc = 20*21 + 4*3
budget_check_con = 20*10 + 4*4

xr = np.linspace(0, 25, 80)
yr = np.linspace(0, 10, 80)
X, Y = np.meshgrid(xr, yr)
Z = P_num(X, Y)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "surface"}, {"type": "surface"}]],
    subplot_titles=("Unconstrained maximum", "Constrained optimum (budget)"),
)

# --- ซ้าย: surface + unconstrained point ---
fig.add_trace(go.Surface(x=X, y=Y, z=Z, colorscale="Viridis", opacity=0.85, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter3d(
    x=[unconstrained_pt[0]], y=[unconstrained_pt[1]], z=[P_unconstrained],
    mode="markers", marker={"size": 6, "color": "blue", "symbol": "diamond"},
    name=f"Unconstrained {unconstrained_pt}, P={P_unconstrained}",
), row=1, col=1)

# --- ขวา: surface + constraint path + both points ---
fig.add_trace(go.Surface(x=X, y=Y, z=Z, colorscale="Viridis", opacity=0.6, showscale=False), row=1, col=2)

y_cons = np.linspace(0, 12, 50)
x_cons = (216 - 4*y_cons) / 20
z_cons = P_num(x_cons, y_cons)
fig.add_trace(go.Scatter3d(
    x=x_cons, y=y_cons, z=z_cons, mode="lines",
    line={"color": "red", "width": 6},
    name="Constraint path: 20x+4y=216",
), row=1, col=2)

fig.add_trace(go.Scatter3d(
    x=[unconstrained_pt[0]], y=[unconstrained_pt[1]], z=[P_unconstrained],
    mode="markers", marker={"size": 6, "color": "blue", "symbol": "diamond"},
    name=f"Unconstrained {unconstrained_pt}, P={P_unconstrained}", showlegend=False,
), row=1, col=2)

fig.add_trace(go.Scatter3d(
    x=[constrained_pt[0]], y=[constrained_pt[1]], z=[P_constrained],
    mode="markers", marker={"size": 6, "color": "green"},
    name=f"Constrained {constrained_pt}, P={P_constrained}",
), row=1, col=2)

fig.add_trace(go.Scatter3d(
    x=[unconstrained_pt[0], constrained_pt[0]],
    y=[unconstrained_pt[1], constrained_pt[1]],
    z=[P_unconstrained, P_constrained],
    mode="lines", line={"color": "purple", "width": 4, "dash": "dash"},
    showlegend=False,
), row=1, col=2)

fig.update_scenes(xaxis_title="x (units of A)", yaxis_title="y (units of B)", zaxis_title="Profit P(x,y)")
fig.update_layout(
    width=1000, height=520,
    legend={"orientation": "h", "y": -0.05},
    margin={"l": 0, "r": 0, "t": 40, "b": 0},
)
fig.show()

print("สรุปเปรียบเทียบ:")
print(f"  -----------------------------------------")
print(f"  UNCONSTRAINED (ถ้าไม่จำกัดงบ):")
print(f"    - ผลิต: x={unconstrained_pt[0]}, y={unconstrained_pt[1]} หน่วย")
print(f"    - กำไร: P={P_unconstrained} พันบาท <- สูงสุด!!")
print(f"    - ใช้งบ: 20({unconstrained_pt[0]})+4({unconstrained_pt[1]}) = {budget_check_unc} <- เกินกำหนด (216)")
print(f"")
print(f"  CONSTRAINED (จริงๆ กับงบจำกัด 216):")
print(f"    - ผลิต: x={constrained_pt[0]}, y={constrained_pt[1]} หน่วย")
print(f"    - กำไร: P={P_constrained} พันบาท <- ดีที่สุดที่ทำได้")
print(f"    - ใช้งบ: 20({constrained_pt[0]})+4({constrained_pt[1]}) = {budget_check_con} (เพียงพอ)")
print(f"")
print(f"  ผลกระทบของ CONSTRAINT:")
print(f"    - ขาดกำไรไป: {P_unconstrained - P_constrained} พันบาท ({100*(P_unconstrained-P_constrained)/P_unconstrained:.1f}%)")
print(f"    - Lagrange multiplier lambda=1 <- ถ้าเพิ่มงบ 1 หน่วย กำไรจะเพิ่ม ~1 พันบาท")
print(f"  -----------------------------------------")


---

## 💡 Key Insights จากกราฟ

### 1️⃣ Contour Plot (Example 3): **Level curves สัมผัส constraint**

**ซ้าย** — ไม่มี constraint:
- ศูนย์สูตร $\nabla f = 0$ ตรง center จุด $(2,3)$ → global min

**ขวา** — มี constraint $x+2y=7$:
- เส้น constraint ตัด level curves หลายเส้น
- ยกเว้น **จุดสัมผัส** $(5,1)$ ที่ level curve ขนานกับ constraint
- **ที่จุดสัมผัส:** gradient $\nabla f$ ตั้งฉากกับ constraint ⇒ $\nabla f \parallel \nabla g$ ✓ ตรงตามเงื่อนไข Lagrange

### 2️⃣ 3D Surface (Example 4): **Constraint path ตามช่องของงบประมาณ**

**ซ้าย** — ถ้าไม่จำกัดงบ:
- Peak ชัดเจนที่ $(21,3)$, $P=648$ (กำไรสูงสุด)

**ขวา** — กับ constraint $20x+4y=216$:
- เส้นแดง (constraint path) เดินขึ้นลงบนพื้นผิว
- **ยอดสูงสุดบนเส้นแดง** คือ constrained optimum $(10,4)$, $P=540$
- ลูกศรสีม่วง: "หากเราเคลื่อนเข้าหา global max แต่ถูกบังคับโดยงบประมาณ"

### 3️⃣ ความหมาย Lagrange Multiplier $\lambda$

| Example | $\lambda$ | ความหมาย |
|---|---|---|
| Example 3 | $\lambda = 8$ | ถ้าเพิ่มงบจาก $(x+2y=7)$ เป็น $(x+2y=8)$ → ค่า $f$ ↓ 8 หน่วย |
| Example 4 | $\lambda = 1$ | ถ้าเพิ่มงบจาก $(20x+4y=216)$ เป็น $\ldots=217$ → $P$ ↑ ≈1 พันบาท |

**💰 นัยยะทางธุรกิจ:** ถ้า $\lambda > $ (ต้นทุนเพิ่มงบ 1 หน่วย) ⇒ **คุ้มที่จะขอเพิ่มงบ!**

---

### 4️⃣ ส่วนประกอบทั้งสามแบบของ optimization

```
┌─────────────────────────────────────────────────────────┐
│                  OPTIMIZATION PROBLEM                    │
├──────────────────┬──────────────────┬──────────────────┤
│  UNCONSTRAINED   │  CONSTRAINED-EQ  │  CONSTRAINED-INQ │
│  optimize f(x)   │ optimize f(x)    │ optimize f(x)    │
│                  │ s.t. g(x)=0      │ s.t. g(x)≤0      │
│                  │                  │ (Linear Program) │
├──────────────────┼──────────────────┼──────────────────┤
│ ∇f = 0           │ ∇f = λ∇g + μ∇h  │ KKT conditions:  │
│ ⇒ Hessian test   │ ⇒ Lagrange       │ ∇f + λ∇g = 0    │
│                  │ ⇒ Bordered H     │ λg = 0, λ≥0      │
└──────────────────┴──────────────────┴──────────────────┘
```

**ที่เรียนมา:** 
- ✅ Unconstrained + Hessian test (Part 2)
- ✅ Constrained equality + Lagrange (Part 3 & 4)  ← **ที่เพิ่งทำกราฟ**
- ❓ Constrained inequality + KKT (ไม่ครอบคลุมในแล็บนี้)
